<a href="https://colab.research.google.com/github/yogesh-bhattarai/Django/blob/main/lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df= pd.read_csv(r'/content/property.csv')

In [2]:
df

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Price
0,1360,2,3,1953,7860,3.039481e+05
1,4272,3,3,1997,5292,8.603863e+05
2,3592,4,1,1983,9723,7.343898e+05
3,966,6,1,1903,4086,2.264488e+05
4,4926,6,4,1944,1081,1.022486e+06
...,...,...,...,...,...,...
999995,2540,3,3,1942,3003,5.317687e+05
999996,851,1,3,1900,8064,1.873590e+05
999997,1200,2,1,1988,7248,2.614827e+05
999998,1580,6,2,1910,5942,3.541821e+05


In [5]:
df.head()

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Price
0,1360,2,3,1953,7860,3.039481e+05
1,4272,3,3,1997,5292,8.603863e+05
2,3592,4,1,1983,9723,7.343898e+05
3,966,6,1,1903,4086,2.264488e+05
4,4926,6,4,1944,1081,1.022486e+06


In [4]:
df.isnull().sum()

,0
Square_Footage,0
Num_Bedrooms,0
Num_Bathrooms,0
Year_Built,0
Lot_Size,0
Price,0


In [6]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [7]:
spark = SparkSession.builder \
    .appName("Property Price Prediction") \
    .getOrCreate()

In [12]:
df = spark.read.csv('/content/property.csv', header=True, inferSchema=True)

In [14]:
df.show()

+--------------+------------+-------------+----------+--------+------------------+
|Square_Footage|Num_Bedrooms|Num_Bathrooms|Year_Built|Lot_Size|             Price|
+--------------+------------+-------------+----------+--------+------------------+
|          1360|           2|            3|      1953|    7860| 303948.1373854071|
|          4272|           3|            3|      1997|    5292| 860386.2685075302|
|          3592|           4|            1|      1983|    9723| 734389.7538956215|
|           966|           6|            1|      1903|    4086| 226448.8070714377|
|          4926|           6|            4|      1944|    1081|1022486.2616704078|
|          3944|           6|            2|      1938|    3542| 845638.1354384426|
|          3671|           2|            1|      1963|    5105| 748779.2192281872|
|          3419|           4|            2|      1925|    5448| 743007.2614135538|
|           630|           2|            2|      2012|    3204| 135656.4528785377|
|   

In [15]:
print("Dataset Preview:")
df.show(5)

Dataset Preview:
+--------------+------------+-------------+----------+--------+------------------+
|Square_Footage|Num_Bedrooms|Num_Bathrooms|Year_Built|Lot_Size|             Price|
+--------------+------------+-------------+----------+--------+------------------+
|          1360|           2|            3|      1953|    7860| 303948.1373854071|
|          4272|           3|            3|      1997|    5292| 860386.2685075302|
|          3592|           4|            1|      1983|    9723| 734389.7538956215|
|           966|           6|            1|      1903|    4086| 226448.8070714377|
|          4926|           6|            4|      1944|    1081|1022486.2616704078|
+--------------+------------+-------------+----------+--------+------------------+
only showing top 5 rows


In [16]:
# Check Schema
df.printSchema()

root
 |-- Square_Footage: integer (nullable = true)
 |-- Num_Bedrooms: integer (nullable = true)
 |-- Num_Bathrooms: integer (nullable = true)
 |-- Year_Built: integer (nullable = true)
 |-- Lot_Size: integer (nullable = true)
 |-- Price: double (nullable = true)



In [17]:
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

In [18]:
models = {
    "Model 1": ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"],
    "Model 2": ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms", "Lot_Size"],
    "Model 3": ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms", "Year_Built"],
    "Model 4": ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms", "Year_Built", "Lot_Size"]
}

In [19]:
results = []

for model_name, feature_cols in models.items():

    print(f"\nRunning {model_name}")
    print("Features:", feature_cols)

    # Create Feature Vector
    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features"
    )

    train_vector = assembler.transform(train_data)
    test_vector = assembler.transform(test_data)

    # Linear Regression Model
    lr = LinearRegression(
        featuresCol="features",
        labelCol="Price"
    )

    lr_model = lr.fit(train_vector)

    # Predictions
    predictions = lr_model.transform(test_vector)

    # Evaluate R²
    evaluator = RegressionEvaluator(
        labelCol="Price",
        predictionCol="prediction",
        metricName="r2"
    )

    r2 = evaluator.evaluate(predictions)

    results.append((model_name, feature_cols, r2))

    print(f"R² Score: {r2:.4f}")

# Display Comparison Results
print("\n==============================")
print("MODEL COMPARISON")
print("==============================")


Running Model 1
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
R² Score: 0.9939

Running Model 2
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Lot_Size']
R² Score: 0.9939

Running Model 3
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built']
R² Score: 0.9941

Running Model 4
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']
R² Score: 0.9941

MODEL COMPARISON


In [20]:
# Display Comparison Results
print("\n==============================")
print("MODEL COMPARISON")
print("==============================")
for model_name, features, r2 in results:
    print(f"{model_name}")
    print(f"Features: {features}")
    print(f"R² Score: {r2:.4f}")
    print("-" * 40)

# Find Best Model
best_model = max(results, key=lambda x: x[2])

print("\nBest Performing Model")
print(f"Model: {best_model[0]}")
print(f"Features: {best_model[1]}")
print(f"R² Score: {best_model[2]:.4f}")

spark.stop()


MODEL COMPARISON
Model 1
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
R² Score: 0.9939
----------------------------------------
Model 2
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Lot_Size']
R² Score: 0.9939
----------------------------------------
Model 3
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built']
R² Score: 0.9941
----------------------------------------
Model 4
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']
R² Score: 0.9941
----------------------------------------

Best Performing Model
Model: Model 4
Features: ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']
R² Score: 0.9941
